In [ ]:
# Aula 07 - Chatbot utilizando o Gemini

# Instalação das bibliotecas
! pip install -q --upgrade pip jedi
! pip install -q llama-index llama-index-readers-file llama-index-llms-google-genai llama-index-embeddings-google-genai

In [ ]:
!pip install nbformat==5.10.4 nbconvert==7.16.6

In [ ]:
import nbformat
import nbconvert

print("nbformat:", nbformat.__version__)
print("nbconvert:", nbconvert.__version__)

In [ ]:
!pip install -U nbformat nbconvert

In [ ]:

# Rode esta celula antes de usar o Gemini com LlamaIndex
%pip install -q  nest-asyncio
import nest_asyncio
nest_asyncio.apply()
import os
from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

# Pega a variavel de ambiente
os.environ['Gemini_chatbot'] = userdata.get('Gemini_chatbot')
api = os.environ['Gemini_chatbot']


In [ ]:
print(api)

In [ ]:
# Cria a variavel chamada llm
llm = GoogleGenAI(
    model='gemini-2.5-flash-lite',
    api_key=api
)

Settings.llm = llm
#Settings.embed_model = embed_model

1) Envie arquivo para a base de conhecimento utilizando a técnica RAG

Cria uma pasta chamada documentos no Colab e envie seus arquivos para lá

In [ ]:
from google.colab import files
import os
os.makedirs("/contents/documentos", exist_ok=True)
print('Pasta criada em /content/documentos')

In [ ]:
# Leitura dos arquivos
from llama_index.core import SimpleDirectoryReader
documentos = SimpleDirectoryReader(input_dir='/content/documentos')

In [ ]:
docs = documentos.load_data()
print(f'Quantidade de documentos carregados {len(docs)}')

In [ ]:
# Exibindo os metadados do documento
print(docs[0].get_metadata_str())

In [ ]:
# Quebrando o documento em pedaços (nodes)
# importando a biblioteca
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter(chunk_size=1200)
# Fazer a divisao do documento carregado com base no chunk size
nodes = node_parser.get_nodes_from_documents(docs, show_progress=True)
print(f'Quantidade de nodes gerados: {len(nodes)}')


In [ ]:
# Configurando o LLM Gemini e o modelo de embeddings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings

llm = GoogleGenAI(
    model='gemini-2.5-flash-lite',
    api_key=api
)

embed_model = GoogleGenAIEmbedding(
    model='gemini-embedding-001',
    api_key=api
)

Settings.llm = llm
Settings.embed_model = embed_model
print('LLM e embeddings configurados')

2) Criando o indice vetorial, esse indice é criado sem o Chroma DB, diretamente em memoria para simplificar a execução no Colab


In [ ]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes, embed_model=embed_model)
print('Indice criado com sucesso')

In [ ]:
from llama_index.core.base.embeddings.base import similarity
# Realizando consultas no Chatbot
# query engine realiza consulta simples no documento
query_engine = index.as_query_engine(llm=llm, similarity_top_k=1)
response = query_engine.query("Quais grãos estao disponiveis")
print(response)

In [ ]:
chat_engine = index.as_chat_engine(llm=llm, similarity_top_k=1)


In [ ]:
response = chat_engine.chat("Qual o valor de cada um")
print(response)

4) Modo Chat com memória resumida


In [ ]:
from llama_index.core.memory import ChatSummaryMemoryBuffer
memory = ChatSummaryMemoryBuffer(llm=llm, token_limit=256)
chat_engine = index.as_chat_engine(
    chat_mode='context',
    llm=llm,
    memory=memory,
    system_prompt=(
        'Você é especialista em cafés da loja Serenatto, uma loja online que vende '
        'grãos de café torrados. Sua função é tirar dúvidas de forma simpática e natural sobre os grãos disponíveis'
    )
)

In [ ]:
response = chat_engine.chat('Olá')
print(response.response)

In [ ]:
response = chat_engine.chat('Voce pode me dar detalhes sobre o Catui amarelo')
print(response.response)

In [ ]:
response = chat_engine.chat('qual o preço')
print(response.response)

In [ ]:
response = chat_engine.chat('Voce pode me dar detalhes sobre o bourbon vermelho')
print(response.response)

In [ ]:
response = chat_engine.chat('qual o preço citado antes')
print(response.response)

In [ ]:
memory.get()

In [ ]:
# Reset do chat
chat_engine.reset()
print('Chat resetado')

In [ ]:
response = chat_engine.chat('O Neymar vai para a copa ')
print(response.response)